# Day 21 — Class imbalance and appropriate evaluation metrics

Run each cell with **Shift + Enter**, from the top. We use `student_data_150.csv` (8 real students and 142 **synthetic** ones) and synthetic data generated in the notebook.

## 1. Understanding class imbalance and its impact

In [ ]:
# Get the student dataset: Colab starts with an empty folder. If the download fails, upload student_data_150.csv
# with the Files panel (the folder icon on the left) and run this cell again
import os
import urllib.request

URL = "https://raw.githubusercontent.com/neerajb1/learnforge-notebooks/main/ai-ml-120/day-021/data/student_data_150.csv"
if not os.path.exists("student_data_150.csv"):
    urllib.request.urlretrieve(URL, "student_data_150.csv")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Our students: how balanced are passing and failing? The 23 test students are set aside first, as every day
students = pd.read_csv("student_data_150.csv")
features = ["StudyHours", "Attendance", "PreviousScore"]
pool, test = train_test_split(students, test_size=0.15, random_state=42, stratify=students["Passed"])
X, y = pool[features], pool["Passed"].astype(int)          # 1 = pass, 0 = fail
for name, part in [("All 150", students), ("Training 127", pool)]:
    counts = part["Passed"].map({True: "pass", False: "fail"}).value_counts()
    print(f"{name:<13} pass {counts['pass']:>3} ({counts['pass'] / len(part):.1%})"
          f"  fail {counts['fail']:>3} ({counts['fail'] / len(part):.1%})"
          f"  ratio 1:{counts['pass'] / counts['fail']:.1f}")

All 150       pass 106 (70.7%)  fail  44 (29.3%)  ratio 1:2.4
Training 127  pass  90 (70.9%)  fail  37 (29.1%)  ratio 1:2.4


In [ ]:
# A highly imbalanced dataset: 1,000 synthetic card transactions, exactly 10 of them fraud (1%).
# Seeded, and not real: fraud here tends to be larger and at night, with plenty of overlap
rng = np.random.default_rng(21)
n, n_fraud = 1000, 10
is_fraud = np.zeros(n, dtype=int)
is_fraud[rng.choice(n, n_fraud, replace=False)] = 1
amount = np.where(is_fraud == 1, rng.lognormal(np.log(6000), 0.9, n), rng.lognormal(np.log(1500), 0.9, n)).round()
hour = np.where(is_fraud == 1, rng.choice([0, 1, 2, 3, 22, 23, 14, 19], n), rng.integers(7, 23, n))
transactions = pd.DataFrame({"TransactionID": np.arange(1, n + 1), "Amount": amount, "Hour": hour, "IsFraud": is_fraud})
print(transactions.head().to_string(index=False))
counts = transactions["IsFraud"].value_counts()
print(f"Legitimate (0): {counts[0]} ({counts[0] / n:.0%}) | fraud (1): {counts[1]} ({counts[1] / n:.0%})")
print(f"Imbalance ratio 1:{counts[0] // counts[1]}")

 TransactionID  Amount  Hour  IsFraud
             1   671.0    21        0
             2   791.0    12        0
             3  1177.0    19        0
             4  2615.0    16        0
             5   984.0    13        0
Legitimate (0): 990 (99%) | fraud (1): 10 (1%)
Imbalance ratio 1:99


In [ ]:
from sklearn.dummy import DummyClassifier

# The accuracy trap, in one line each: a "model" that always predicts the majority class
always_legit = DummyClassifier(strategy="most_frequent").fit(transactions[["Amount", "Hour"]], transactions["IsFraud"])
pred = always_legit.predict(transactions[["Amount", "Hour"]])
print(f"Transactions: accuracy {(pred == transactions['IsFraud']).mean():.3f},"
      f" frauds caught {((pred == 1) & (transactions['IsFraud'] == 1)).sum()} of 10")
always_pass = DummyClassifier(strategy="most_frequent").fit(X, y)
pred = always_pass.predict(X)
print(f"Students:     accuracy {(pred == y).mean():.3f}, failing students caught {((pred == 0) & (y == 0)).sum()} of {(y == 0).sum()}")

Transactions: accuracy 0.990, frauds caught 0 of 10
Students:     accuracy 0.709, failing students caught 0 of 37


In [ ]:
from sklearn.linear_model import LogisticRegression

# Impact on learning: the same two overlapping groups, only the minority's share changes.
# Logistic regression, trained on 75%, checked on the other 25% of each synthetic dataset
def overlapping_groups(minority_share, n=4000, seed=0):
    r = np.random.default_rng(seed)
    labels = (r.random(n) < minority_share).astype(int)
    points = r.normal(0, 1, (n, 2)) + labels[:, None] * 1.5      # the minority sits 1.5 units away: overlap
    return points, labels

print("minority  accuracy  minority recall  minority predicted / actual")
for share in [0.5, 0.2, 0.05, 0.01]:
    P, L = overlapping_groups(share)
    P_tr, P_va, L_tr, L_va = train_test_split(P, L, test_size=0.25, stratify=L, random_state=0)
    p = LogisticRegression().fit(P_tr, L_tr).predict(P_va)
    recall = ((p == 1) & (L_va == 1)).sum() / (L_va == 1).sum()
    print(f"{share:>8.0%}  {(p == L_va).mean():>8.3f}  {recall:>15.3f}  {(p == 1).sum():>9} / {(L_va == 1).sum()}")

minority  accuracy  minority recall  minority predicted / actual
     50%     0.860            0.854        500 / 508
     20%     0.898            0.702        186 / 208
      5%     0.966            0.453         29 / 53
      1%     0.990            0.000          0 / 10


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# On our students: every training student predicted by a model that didn't train on them (5-fold CV)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = Pipeline([("scaler", StandardScaler()), ("classifier", LogisticRegression(max_iter=5000))])
pred = cross_val_predict(model, X, y, cv=cv)
cm = pd.crosstab(y.map({0: "Actual fail", 1: "Actual pass"}), pd.Series(pred, index=y.index).map({0: "Pred fail", 1: "Pred pass"}))
cm.index.name, cm.columns.name = None, None
print(cm.to_string())
print(f"Accuracy {(pred == y).mean():.3f}")
print(f"Passing students found: {((pred == 1) & (y == 1)).sum() / (y == 1).sum():.1%}")
print(f"Failing students found: {((pred == 0) & (y == 0)).sum() / (y == 0).sum():.1%}")

             Pred fail  Pred pass
Actual fail         22         15
Actual pass         11         79
Accuracy 0.795
Passing students found: 87.8%
Failing students found: 59.5%


## 2. Why accuracy becomes misleading

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

# The accuracy trap at scale: 10,000 transactions, 9,900 legitimate (0) and 100 fraud (1); predict "legitimate" for all
actual = np.array([0] * 9900 + [1] * 100)
predicted = np.zeros_like(actual)
tn, fp, fn, tp = confusion_matrix(actual, predicted, labels=[0, 1]).ravel()
print("               Pred legitimate  Pred fraud")
print(f"Actual legit   {tn:>15,}  {fp:>10,}")
print(f"Actual fraud   {fn:>15,}  {tp:>10,}")
print(f"Accuracy {accuracy_score(actual, predicted):.1%} | frauds detected {tp} of {tp + fn}")

               Pred legitimate  Pred fraud
Actual legit             9,900           0
Actual fraud               100           0
Accuracy 99.0% | frauds detected 0 of 100


In [ ]:
# Compare two models on the same 10,000 transactions (the mock's counts). FP = legitimate flagged as fraud
models_ab = {"A": {"TN": 9850, "TP": 20, "FN": 80}, "B": {"TN": 9700, "TP": 80, "FN": 20}}
for m in models_ab.values():
    m["FP"] = 9900 - m["TN"]
    m["accuracy"] = (m["TP"] + m["TN"]) / 10000
print("model  accuracy  fraud caught  fraud missed  false alarms")
for name, m in models_ab.items():
    print(f"{name:<5}  {m['accuracy']:>8.1%}  {m['TP']:>12}  {m['FN']:>12}  {m['FP']:>12}")
# Which is cheaper depends on the cost of each error. Cost = missed frauds x cost_miss + false alarms x cost_alarm
print("cost of a miss / cost of a false alarm -> cheaper model")
for ratio in [1, 2, 2.5, 3, 10, 50]:
    cost = {n: m["FN"] * ratio + m["FP"] for n, m in models_ab.items()}
    verdict = "tie" if cost["A"] == cost["B"] else min(cost, key=cost.get)
    print(f"{ratio:>5} x  A {cost['A']:>6,.0f}  B {cost['B']:>6,.0f}  -> {verdict}")

model  accuracy  fraud caught  fraud missed  false alarms
A         98.7%            20            80            50
B         97.8%            80            20           200
cost of a miss / cost of a false alarm -> cheaper model
    1 x  A    130  B    220  -> A
    2 x  A    210  B    240  -> A
  2.5 x  A    250  B    250  -> tie
    3 x  A    290  B    260  -> B
   10 x  A    850  B    400  -> B
   50 x  A  4,050  B  1,200  -> B


In [ ]:
# Majority-only accuracy as imbalance grows: a DummyClassifier fitted on each distribution
distributions = {"Balanced 50:50": (500, 500), "Moderate 90:10": (900, 100),
                 "High 99:1": (990, 10), "Extreme 999:1": (999, 1)}
print("distribution     accuracy  minority found")
for name, (majority, minority) in distributions.items():
    labels = np.array([0] * majority + [1] * minority)
    dummy = DummyClassifier(strategy="most_frequent").fit(np.zeros((len(labels), 1)), labels)
    pred = dummy.predict(np.zeros((len(labels), 1)))
    print(f"{name:<15} {accuracy_score(labels, pred):>9.1%}  {((pred == 1) & (labels == 1)).sum():>6} of {minority}")

distribution     accuracy  minority found
Balanced 50:50      50.0%       0 of 500
Moderate 90:10      90.0%       0 of 100
High 99:1           99.0%       0 of 10
Extreme 999:1       99.9%       0 of 1


In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Real models on Section 1's 1,000 transactions (10 frauds), 5-fold stratified CV: 2 frauds per validation fold
Xt, yt = transactions[["Amount", "Hour"]], transactions["IsFraud"]
cv_t = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
candidates = {
    "Logistic regression": Pipeline([("scaler", StandardScaler()), ("classifier", LogisticRegression())]),
    "Decision tree (depth 3)": DecisionTreeClassifier(max_depth=3, random_state=42),
    "Logistic, balanced": Pipeline([("scaler", StandardScaler()),
                                    ("classifier", LogisticRegression(class_weight="balanced"))]),
}
print("model                    accuracy  fraud caught  false alarms")
for name, m in candidates.items():
    p = cross_val_predict(m, Xt, yt, cv=cv_t)
    print(f"{name:<24} {accuracy_score(yt, p):>8.3f}  {((p == 1) & (yt == 1)).sum():>6} of 10  {((p == 1) & (yt == 0)).sum():>12}")

model                    accuracy  fraud caught  false alarms


Logistic regression         0.989       1 of 10             2
Decision tree (depth 3)     0.993       4 of 10             1
Logistic, balanced          0.883       9 of 10           116


In [ ]:
# Our students, with "fail" as the positive class (the one we want to find). Cross-validated predictions
student_models = {
    "Always predict pass": DummyClassifier(strategy="most_frequent"),
    "Logistic regression": model,
    "Logistic, balanced": Pipeline([("scaler", StandardScaler()),
                                    ("classifier", LogisticRegression(class_weight="balanced", max_iter=5000))]),
}
fail = (y == 0)
print("model                 accuracy   TP   FN   FP   TN")
for name, m in student_models.items():
    p_fail = cross_val_predict(m, X, y, cv=cv) == 0
    tp, fn = (p_fail & fail).sum(), (~p_fail & fail).sum()
    fp, tn = (p_fail & ~fail).sum(), (~p_fail & ~fail).sum()
    print(f"{name:<21} {(tp + tn) / len(y):>8.3f} {tp:>4} {fn:>4} {fp:>4} {tn:>4}")

model                 accuracy   TP   FN   FP   TN
Always predict pass      0.709    0   37    0   90
Logistic regression      0.795   22   15   11   79
Logistic, balanced       0.780   29    8   20   70
